# ============================================
# Cell 1: Import libraries and set up global variables for temperature and wind
# ============================================

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
# Global file paths and parameters for temperature/wind analysis
BASE_DIR = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart'
TEMP_MERGED_FILE = os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.merged.nc')
WIND_MERGED_FILE = os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.merged.nc')
LAT = 60
PLEV_TEMP = 5000   # 5000 Pa (50 hPa)
PLEV_WIND = 1000   # 1000 Pa (10 hPa)

# ============================================
# Cell 2: Define temperature data processing functions
# ============================================

In [ ]:
def process_temp_data(file_pattern, plev=5000, lat_range=(60, 90)):
    """
    Process WACCM temperature data.
    Reads nc files matching file_pattern, computes zonal mean,
    selects specified pressure level and latitude range,
    then takes the minimum along latitude.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    temp = ds['T'].mean(dim='lon')
    temp = temp.sel(plev=plev)
    temp_min = temp.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    return temp_min

def process_temp_base_data(file_path, plev=5000, lat_range=(60, 90), months=[1,2,3,4,5,6]):
    """
    Process baseline temperature data.
    Returns a tuple (base, climatology), where base is the data for year 8 and
    climatology is the daily mean climatology.
    """
    ds = xr.open_dataset(file_path)
    temp = ds['T'].sel(plev=plev)
    temp = temp.sel(time=ds.time.dt.month.isin(months))
    temp = temp.mean(dim='lon')
    temp_min = temp.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    base = temp_min.sel(time=temp_min.time.dt.year==8)
    clim = temp_min.groupby("time.dayofyear").mean()
    return base, clim


# ============================================
# Cell 3: Define plotting functions for temperature evolution
# ============================================

In [ ]:
def plot_waccm_results(base_temp, clim_temp, jan_temp, feb_temp, mar_temp, save_path, plev=5000, lat=60):
    """
    Plot WACCM temperature evolution.
    Plots the base (black solid), climatology (black dashed), and monthly mean curves for Jan, Feb, and Mar.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    ax.plot(np.arange(len(base_temp)), base_temp, color='black', linewidth=3, label='Base')
    ax.plot(np.arange(len(clim_temp)), clim_temp, color='black', linestyle=':', linewidth=3, label='Clim')
    ax.plot(np.arange(len(jan_temp.time)), jan_temp.mean(dim='member'), color='forestgreen', linewidth=3, label='Jan')
    ax.plot(np.arange(31, 31+len(feb_temp.time)), feb_temp.mean(dim='member'), color='royalblue', linewidth=3, label='Feb')
    ax.plot(np.arange(59, 59+len(mar_temp.time)), mar_temp.mean(dim='member'), color='hotpink', linewidth=3, label='Mar')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Minimum Temperature, {lat}°N, {plev/100:.0f} hPa (K)')
    ax.legend()
    
    plt.savefig(save_path + '.pdf')
    plt.savefig(save_path + '.png')
    return fig, ax

def plot_temp_february_perturbations(base_temp, feb_small, feb_large, feb_q, feb_nocoupl, save_path, plev=5000, lat_range=(60, 90)):
    """
    Plot February perturbation temperature evolution.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    ax.plot(np.arange(len(base_temp)), base_temp, color='black', linewidth=3, label='Base')
    ax.plot(np.arange(len(base_temp)), base_temp, color='black', linestyle=':', linewidth=3, label='Clim')
    
    x_feb = np.arange(31, 31+len(feb_small.time))
    ax.plot(x_feb, feb_small.mean(dim='member'), color='navy', linewidth=3, label='Feb, small pert')
    ax.fill_between(x_feb, feb_small.min(dim='member'), feb_small.max(dim='member'), color='navy', alpha=0.3)
    
    x_feb2 = np.arange(31, 31+len(feb_large.time))
    ax.plot(x_feb2, feb_large.mean(dim='member'), color='forestgreen', linewidth=3, label='Feb, large pert')
    ax.fill_between(x_feb2, feb_large.min(dim='member'), feb_large.max(dim='member'), color='forestgreen', alpha=0.3)
    
    x_feb3 = np.arange(31, 31+len(feb_q.time))
    ax.plot(x_feb3, feb_q.mean(dim='member'), color='darkorange', linewidth=3, label='Feb, Q pert')
    ax.fill_between(x_feb3, feb_q.min(dim='member'), feb_q.max(dim='member'), color='darkorange', alpha=0.3)
    
    x_feb_nc = np.arange(31, 31+len(feb_nocoupl.time))
    ax.plot(x_feb_nc, feb_nocoupl.mean(dim='member'), color='darkred', linewidth=3, label='Feb, NOCOUPL')
    ax.fill_between(x_feb_nc, feb_nocoupl.min(dim='member'), feb_nocoupl.max(dim='member'), color='darkred', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev/100:.0f} hPa (K)')
    ax.legend()
    
    plt.savefig(save_path + '.pdf')
    plt.savefig(save_path + '.png')
    return fig, ax

def plot_temp_march_perturbations(base_temp, mar_small, mar_large, mar_q, save_path, plev=5000, lat_range=(60, 90)):
    """
    Plot March perturbation temperature evolution.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    ax.plot(np.arange(len(base_temp)), base_temp, color='black', linewidth=3, label='Base')
    ax.plot(np.arange(len(base_temp)), base_temp, color='black', linestyle=':', linewidth=3, label='Clim')
    
    x_mar = np.arange(59, 59+len(mar_small.time))
    ax.plot(x_mar, mar_small.mean(dim='member'), color='crimson', linewidth=3, label='Mar, small pert')
    ax.fill_between(x_mar, mar_small.min(dim='member'), mar_small.max(dim='member'), color='crimson', alpha=0.3)
    
    x_mar2 = np.arange(59, 59+len(mar_large.time))
    ax.plot(x_mar2, mar_large.mean(dim='member'), color='indigo', linewidth=3, label='Mar, large pert')
    ax.fill_between(x_mar2, mar_large.min(dim='member'), mar_large.max(dim='member'), color='indigo', alpha=0.3)
    
    x_mar3 = np.arange(59, 59+len(mar_q.time))
    ax.plot(x_mar3, mar_q.mean(dim='member'), color='teal', linewidth=3, label='Mar, Q pert')
    ax.fill_between(x_mar3, mar_q.min(dim='member'), mar_q.max(dim='member'), color='teal', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev/100:.0f} hPa (K)')
    ax.legend()
    
    plt.savefig(save_path + '.pdf')
    plt.savefig(save_path + '.png')
    return fig, ax


# ============================================
# Cell 4: Define wind data processing function and baseline processing for wind
# ============================================

In [ ]:
def process_waccm_data(file_pattern, lat=60, plev=1000):
    """
    Process WACCM wind data.
    Computes zonal mean and selects a given pressure level and nearest latitude.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    wind = ds['U'].mean(dim='lon')
    wind = wind.sel(plev=plev)
    wind_sel = wind.sel(lat=lat, method='nearest')
    return wind_sel

def process_base_data(file_path, lat=60, plev=1000, months=[1,2,3,4,5,6]):
    """
    Process baseline wind data.
    Returns a tuple (base, climatology) from the merged file.
    """
    ds = xr.open_dataset(file_path)
    wind = ds['U'].sel(plev=plev)
    wind = wind.sel(time=ds.time.dt.month.isin(months))
    wind = wind.mean(dim='lon')
    wind_sel = wind.sel(lat=lat, method='nearest')
    base = wind_sel.sel(time=wind_sel.time.dt.year==8)
    clim = wind_sel.groupby("time.dayofyear").mean()
    return base, clim

# ============================================
# Cell 5: Read and preprocess temperature and wind data for 008
# ============================================

In [ ]:
# Temperature data
temp_base, temp_clim = process_temp_base_data(TEMP_MERGED_FILE, plev=PLEV_TEMP, lat_range=(60,90))
jan_temp = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))
feb_temp = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))
mar_temp = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))

# February perturbation experiments
feb_temp_2 = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))
feb_temp_3 = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))
feb_temp_nc = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))

mar_temp_2 = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))
mar_temp_3 = process_temp_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'), plev=PLEV_TEMP, lat_range=(60,90))

# Wind data
wind_base, wind_clim = process_base_data(WIND_MERGED_FILE, lat=LAT, plev=PLEV_WIND)
jan_wind = process_waccm_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'), lat=LAT, plev=PLEV_WIND)
feb_wind = process_waccm_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'), lat=LAT, plev=PLEV_WIND)
mar_wind = process_waccm_data(os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'), lat=LAT, plev=PLEV_WIND)

# ============================================
# Cell 6: Plot temperature and wind evolution for 008
# ============================================

In [ ]:
# Temperature evolution plot
temp_save_path = '/home/weiji/restart_exam/plots/T_min_evolution_min_restart_0008'
fig_temp, ax_temp = plot_waccm_results(temp_base, temp_clim, jan_temp, feb_temp, mar_temp, temp_save_path, plev=PLEV_TEMP, lat=LAT)

# February perturbation temperature plot
temp_feb_save = '/home/weiji/restart_exam/plots/T_min_evolution_min_restart_0008_Feb'
fig_temp_feb, ax_temp_feb = plot_temp_february_perturbations(temp_base, feb_temp, feb_temp_2, feb_temp_3, feb_temp_nc, temp_feb_save, plev=PLEV_TEMP, lat_range=(60,90))

# March perturbation temperature plot
temp_mar_save = '/home/weiji/restart_exam/plots/T_min_evolution_min_restart_0008_Mar'
fig_temp_mar, ax_temp_mar = plot_temp_march_perturbations(temp_base, mar_temp, mar_temp_2, mar_temp_3, temp_mar_save, plev=PLEV_TEMP, lat_range=(60,90))

# Wind evolution plot
wind_save_path = '/home/weiji/restart_exam/plots/U_evolution_min_restart_0008'
fig_wind, ax_wind = plot_waccm_results(wind_base, wind_clim, jan_wind, feb_wind, mar_wind, wind_save_path, plev=PLEV_WIND, lat=LAT)

plt.close('all')